# Climate MVT: Cloud-Optimized GeoTIFF Generation

This notebook generates Cloud-Optimized GeoTIFF (COG) files for climate data visualization.

**What this does:**
1. Load climate grid data from PostgreSQL
2. Compute global historical min/max across all times
3. Generate progressive zoom levels (z0-z10) with increasing resolution
4. Apply green colormap (dark → light) to Band 1 (visual)
5. Normalize to grayscale for Band 2 (raw data)
6. Embed WGS84 georeferencing and metadata
7. Save COGs to `/local_only/climate_mvt/{variable}/{time}/z{0-10}.tif`

**Output:**
- Dual-band GeoTIFF files ready for DeckGL BitmapLayer consumption
- Band 1: RGB (colormapped, visual)
- Band 2: Grayscale (raw data, 0-255 normalized)
- Full GeoTIFF metadata tags embedded

## Setup & Configuration

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
import rasterio
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

In [ ]:
# Database configuration
DB_PARAMS = {
    "host": "localhost",
    "user": "leon",
    "password": "leon",
    "database": "oc-database",
    "port": 5432,
}

# Output configuration
COG_BASE_DIR = Path.cwd().parent / "local_only" / "climate_mvt"
COG_BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Database: {DB_PARAMS['database']}")
print(f"✓ COG output directory: {COG_BASE_DIR}")

# Australia bounds (approximate, WGS84)
AUSTRALIA_BOUNDS = {
    "west": 112.85,
    "south": -43.65,
    "east": 154.0,
    "north": -10.0,
}

print(f"✓ Bounds: {AUSTRALIA_BOUNDS}")

In [ ]:
# Zoom level specifications
ZOOM_SPECS = {
    0: (256, 256),
    1: (512, 512),
    2: (1024, 1024),
    3: (2048, 2048),
    4: (4096, 4096),
    5: (8192, 8192),
    6: (16384, 16384),
    7: (32768, 32768),
    8: (65536, 65536),
    9: (131072, 131072),
    10: (262144, 262144),
}

print(f"✓ Zoom levels configured: z0 (256×256) through z10 (262144×262144)")

# Green colormap: dark green (#1B5E20) → light green (#C8E6C9)
GREEN_COLORMAP = mcolors.LinearSegmentedColormap.from_list(
    'climate_green',
    ['#1B5E20', '#66BB6A', '#C8E6C9'],
    N=256
)

print(f"✓ Green colormap created")

## Database Access Functions

In [ ]:
@contextmanager
def get_db_cursor():
    """Context manager for database cursor."""
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

def get_available_variables():
    """Get all unique variables in the database."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT DISTINCT variable FROM climate_grid ORDER BY variable"
        )
        return [row['variable'] for row in cur.fetchall()]

def get_available_times(variable):
    """Get all unique times for a variable."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT DISTINCT time FROM climate_grid WHERE variable = %s ORDER BY time",
            [variable]
        )
        return [row['time'] for row in cur.fetchall()]

def get_grid_data(variable, time):
    """Load grid data for a variable and time."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT lat, lon, value FROM climate_grid WHERE variable = %s AND time::text LIKE %s ORDER BY lat, lon",
            [variable, f"{time}%"]
        )
        rows = cur.fetchall()
        
        if not rows:
            return None
        
        lats = np.array([row['lat'] for row in rows])
        lons = np.array([row['lon'] for row in rows])
        values = np.array([row['value'] for row in rows])
        
        return {
            'lats': lats,
            'lons': lons,
            'values': values,
        }

def get_global_min_max(variable):
    """Get global min/max across all times for a variable."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT MIN(value) as min_val, MAX(value) as max_val FROM climate_grid WHERE variable = %s",
            [variable]
        )
        row = cur.fetchone()
        return float(row['min_val']), float(row['max_val'])

print("✓ Database functions defined")

## COG Generation Functions

In [ ]:
def create_green_colormap_rgb(values_normalized):
    """
    Apply green colormap to normalized values (0-1).
    
    Args:
        values_normalized: numpy array with values in range [0, 1]
        
    Returns:
        (H, W, 3) RGB array with values in [0, 255]
    """
    # Apply colormap (returns RGBA)
    rgba = GREEN_COLORMAP(values_normalized)
    # Convert to 8-bit RGB
    rgb = (rgba[:, :, :3] * 255).astype(np.uint8)
    return rgb

def normalize_values(values, data_min, data_max):
    """
    Normalize values to range [0, 1] using global min/max.
    
    Args:
        values: numpy array of raw values
        data_min: global minimum
        data_max: global maximum
        
    Returns:
        (normalized 0-1, grayscale 0-255)
    """
    if data_max == data_min:
        # Handle edge case where all values are the same
        normalized = np.full_like(values, 0.5, dtype=float)
        grayscale = np.full_like(values, 127, dtype=np.uint8)
    else:
        normalized = (values - data_min) / (data_max - data_min)
        normalized = np.clip(normalized, 0, 1)
        grayscale = (normalized * 255).astype(np.uint8)
    
    return normalized, grayscale

def rasterize_grid_data(lats, lons, values, width, height, bounds):
    """
    Rasterize irregularly-spaced grid data to a regular pixel grid.
    
    Args:
        lats, lons, values: numpy arrays of grid points
        width, height: pixel dimensions
        bounds: dict with 'west', 'east', 'south', 'north' keys
        
    Returns:
        (H, W) rasterized grid
    """
    # Create pixel grid
    pixels = np.full((height, width), np.nan, dtype=float)
    
    # Calculate pixel size
    pixel_width = (bounds['east'] - bounds['west']) / width
    pixel_height = (bounds['north'] - bounds['south']) / height
    
    # Map each grid point to nearest pixel
    for lat, lon, val in zip(lats, lons, values):
        if np.isnan(val):
            continue
        
        # Convert geo coords to pixel indices
        x = int((lon - bounds['west']) / pixel_width)
        y = int((bounds['north'] - lat) / pixel_height)
        
        # Check bounds
        if 0 <= x < width and 0 <= y < height:
            pixels[y, x] = val
    
    # Simple interpolation: fill NaNs with nearest neighbor
    # For now, keep NaNs (they'll be skipped in COG)
    
    return pixels

print("✓ Colormap and rasterization functions defined")

In [ ]:
def create_and_save_cog(variable, time, zoom_level, rasterized_data, data_min, data_max, bounds):
    """
    Create a Cloud-Optimized GeoTIFF with dual bands: RGB visual + grayscale raw.
    
    Args:
        variable: Variable name
        time: Time step
        zoom_level: Zoom level (0-10)
        rasterized_data: (H, W) numpy array with grid data
        data_min, data_max: Global min/max for normalization
        bounds: dict with geographic bounds
        
    Returns:
        Path to saved file
    """
    height, width = rasterized_data.shape
    
    # Normalize data
    normalized, grayscale = normalize_values(rasterized_data, data_min, data_max)
    
    # Create RGB band (green colormap)
    rgb_band = create_green_colormap_rgb(normalized)
    
    # Prepare output directory
    output_dir = COG_BASE_DIR / variable / str(time)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"z{zoom_level}.tif"
    
    # Create geotransform
    transform = from_bounds(
        bounds['west'],
        bounds['south'],
        bounds['east'],
        bounds['north'],
        width,
        height
    )
    
    # Create Cloud-Optimized GeoTIFF with two bands
    # Band 1: RGB (visual, colormapped)
    # Band 2: Grayscale (raw data, normalized 0-255)
    
    # We'll use rasterio to create a multi-band GeoTIFF
    # Note: Single GeoTIFF with RGB + grayscale band
    profile = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': 4,  # RGB (3 bands) + raw data (1 band) = 4 bands total
        'dtype': rasterio.uint8,
        'crs': 'EPSG:4326',
        'transform': transform,
        'compress': 'lzw',  # Cloud-optimized compression
        'TILED': 'YES',
        'BLOCKXSIZE': 512,
        'BLOCKYSIZE': 512,
    }
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        # Write RGB bands (Band 1-3)
        dst.write(rgb_band[:, :, 0], 1)  # Red
        dst.write(rgb_band[:, :, 1], 2)  # Green
        dst.write(rgb_band[:, :, 2], 3)  # Blue
        
        # Write grayscale raw data band (Band 4)
        dst.write(grayscale, 4)
        
        # Set colorinterp for RGB bands
        dst.write_colormap(1, {0: (0, 0, 0)})
        
        # Add metadata tags
        tags = {
            'VARIABLE': variable,
            'TIME': str(time),
            'ZOOM_LEVEL': str(zoom_level),
            'DATA_MIN': str(data_min),
            'DATA_MAX': str(data_max),
            'COLORMAP_TYPE': 'green_scale',
            'CRS': 'EPSG:4326',
        }
        dst.update_tags(**tags)
    
    return output_path

print("✓ COG creation function defined")

## User Input

In [ ]:
# Get available variables
print("Fetching available variables from database...")
available_variables = get_available_variables()
print(f"\nAvailable variables: {available_variables}")

# User selects variable
selected_variable = available_variables[0]  # Default to first variable
print(f"\n✓ Selected variable: {selected_variable}")

In [ ]:
# Get available times for selected variable
print(f"Fetching available times for '{selected_variable}'...")
available_times = get_available_times(selected_variable)
print(f"Available times: {len(available_times)} entries")
print(f"First 5: {available_times[:5]}")
print(f"Last 5: {available_times[-5:]}")

# User selects time
selected_time = str(available_times[0])[:10]  # Use first time, extract date portion
print(f"\n✓ Selected time: {selected_time}")

## Load & Validate Data

In [ ]:
print(f"Loading grid data for {selected_variable} at {selected_time}...")
grid_data = get_grid_data(selected_variable, selected_time)

if grid_data is None:
    print(f"ERROR: No data found for {selected_variable} at {selected_time}")
else:
    lats = grid_data['lats']
    lons = grid_data['lons']
    values = grid_data['values']
    
    print(f"✓ Loaded {len(values)} grid points")
    print(f"  Latitude range: {lats.min():.2f} to {lats.max():.2f}")
    print(f"  Longitude range: {lons.min():.2f} to {lons.max():.2f}")
    print(f"  Value range: {values.min():.2f} to {values.max():.2f}")
    print(f"  NaN count: {np.isnan(values).sum()}")

## Compute Global Min/Max

In [ ]:
print(f"Computing global min/max across all times for '{selected_variable}'...")
global_min, global_max = get_global_min_max(selected_variable)

print(f"✓ Global range: {global_min:.2f} to {global_max:.2f}")
print(f"  This will be used for normalizing all COGs for this variable")

## Generate Progressive COGs (z0-z10)

In [ ]:
print(f"\nGenerating COGs for {selected_variable} / {selected_time}")
print(f"Global range: {global_min:.2f} to {global_max:.2f}")
print("\n" + "="*60)

generated_files = []

for zoom_level in sorted(ZOOM_SPECS.keys()):
    width, height = ZOOM_SPECS[zoom_level]
    
    print(f"\nz{zoom_level}: Generating {width}×{height} COG...")
    
    try:
        # Rasterize grid data to pixel grid
        print(f"  - Rasterizing grid data...")
        rasterized = rasterize_grid_data(
            lats, lons, values, width, height, AUSTRALIA_BOUNDS
        )
        print(f"  - Rasterized shape: {rasterized.shape}, NaN count: {np.isnan(rasterized).sum()}")
        
        # Create and save COG
        print(f"  - Creating Cloud-Optimized GeoTIFF...")
        output_path = create_and_save_cog(
            selected_variable,
            selected_time,
            zoom_level,
            rasterized,
            global_min,
            global_max,
            AUSTRALIA_BOUNDS
        )
        
        # Get file size
        file_size_mb = output_path.stat().st_size / (1024**2)
        print(f"  ✓ Saved: {output_path.name} ({file_size_mb:.2f} MB)")
        generated_files.append(output_path)
        
    except Exception as e:
        print(f"  ✗ ERROR: {e}")

print("\n" + "="*60)
print(f"\n✓ Generated {len(generated_files)} COG files")

## Verification & Summary

In [ ]:
print("\nGenerated Files Summary:")
print("="*60)

output_dir = COG_BASE_DIR / selected_variable / str(selected_time)
if output_dir.exists():
    files = sorted(output_dir.glob('z*.tif'))
    total_size_mb = 0
    
    for f in files:
        size_mb = f.stat().st_size / (1024**2)
        total_size_mb += size_mb
        print(f"{f.name:12} {size_mb:8.2f} MB")
    
    print("="*60)
    print(f"{'Total':12} {total_size_mb:8.2f} MB")
    print(f"Location: {output_dir}")
else:
    print("ERROR: Output directory not found!")

In [ ]:
# Test file properties
print("\nVerifying first COG file (z0)...")

test_file = COG_BASE_DIR / selected_variable / str(selected_time) / "z0.tif"

if test_file.exists():
    with rasterio.open(test_file) as src:
        print(f"✓ CRS: {src.crs}")
        print(f"✓ Size: {src.width}×{src.height}")
        print(f"✓ Bands: {src.count}")
        print(f"✓ Data type: {src.dtypes[0]}")
        print(f"✓ Bounds: {src.bounds}")
        print(f"✓ Metadata tags:")
        tags = src.tags()
        for key, value in tags.items():
            print(f"    {key}: {value}")
else:
    print(f"ERROR: Test file not found: {test_file}")

## Next Steps

The COGs have been generated successfully! 

**To use them:**
1. Start the FastAPI server: `python -m uvicorn app.main:app --reload`
2. Test the endpoints:
   - GET `/climate-mvt/variables` - List available variables
   - GET `/climate-mvt/times/{variable}` - List available times
   - GET `/climate-mvt/{variable}/{time}/z{zoom}.tif` - Download COG
3. In your DeckGL frontend, use BitmapLayer to consume the COGs

**To generate more COGs:**
- Change `selected_variable` and/or `selected_time` above and re-run the generation cells
- The notebook will generate all z0-z10 levels for each variable/time combination